# 🤖 Estudo Prático de Agentes de IA com a API do Google Gemini

Este notebook faz parte dos meus estudos em **Inteligência Artificial e Engenharia de Prompt**, focado na criação de um **Sistema Multi-Agente** encadeado para automação de criação de conteúdo (posts de Instagram sobre tecnologia).

### 🎯 Objetivos de Aprendizado:
- Entender o conceito de **Agentes Especialistas** (Buscador, Planejador, Redator e Revisor).
- Utilizar a nova SDK oficial `google-genai` do Gemini.
- Aplicar **Google Search Grounding** para conectar o agente a buscas web em tempo real.
- Encadear as saídas de cada agente como entrada para o próximo na pipeline.
- Lidar com resiliência de rede (retry e fallback) em requisições de LLM.

## 1. Instalação e Configuração das Dependências

In [1]:
%pip install -q google-genai python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import time
from datetime import date
from google import genai
from google.genai import types

# Configuração do cliente Gemini SDK
# Aqui definimos o modelo a ser usado e o fallback dele, ou seja, outro modelo caso ele esteja
# ocupado demais no momento
client = genai.Client()
MODEL_ID = "gemini-2.5-flash"
FALLBACK_MODEL_ID = "gemini-3.5-flash-lite"

def chamar_gemini_com_retry(contents, config=None, max_retries=3):
    """Função auxiliar para tratar picos temporários de alta demanda (Erro 503 / 429)."""
    modelos = [MODEL_ID, FALLBACK_MODEL_ID]
    for m in modelos:
        for tentativa in range(max_retries):
            try:
                res = client.models.generate_content(model=m, contents=contents, config=config)
                return res.text
            except Exception as e:
                err_msg = str(e)
                if "503" in err_msg or "UNAVAILABLE" in err_msg or "429" in err_msg:
                    tempo_espera = (tentativa + 1) * 2
                    print(f"⚠️ Servidor ocupado ({m}). Tentando novamente em {tempo_espera}s...")
                    time.sleep(tempo_espera)
                else:
                    raise e
    raise RuntimeError("Servidor indisponível após várias tentativas.")

## 2. Definição dos Agentes Especialistas

O sistema é composto por 4 agentes que trabalham em sequência:

### Agente 1: Buscador de Notícias (com Google Search Grounding)

In [3]:
def agente_buscador(topico: str, data_de_hoje: str) -> str:
    """Busca informações e novidades recentes no Google sobre o tema."""
    prompt = f"""
    Você é um pesquisador especialista em tendências de tecnologia.
    Pesquise no Google sobre o seguinte tópico: {topico}
    Considere a data atual como: {data_de_hoje}
    
    Resuma as notícias, lançamentos e fatos mais relevantes sobre este assunto.
    """
    
    # Ativa a ferramenta de busca do Google (Search Grounding)
    config = types.GenerateContentConfig(
        tools=[types.Tool(google_search=types.GoogleSearch())]
    )
    
    return chamar_gemini_com_retry(contents=prompt, config=config)

### Agente 2: Planejador de Conteúdo

In [4]:
def agente_planejador(topico: str, lancamentos_buscados: str) -> str:
    """Cria um roteiro estruturado com base nas notícias encontradas."""
    prompt = f"""
    Você é um estrategista de conteúdo para redes sociais.
    Tópico: {topico}
    Pesquisas encontradas:
    {lancamentos_buscados}
    
    Crie um plano estruturado para um post de Instagram contendo:
    1. Gancho (Hook): Frase marcante para prender a atenção.
    2. Pontos Chave: 2 a 3 tópicos centrais a serem explicados.
    3. Chamada para Ação (CTA): Pergunta engajadora ao final.
    """
    
    return chamar_gemini_com_retry(contents=prompt)

### Agente 3: Redator Criativo

In [5]:
def agente_redator(topico: str, plano_de_post: str) -> str:
    """Escreve a primeira versão do post com emojis e hashtags."""
    prompt = f"""
    Você é um Redator Criativo especialista em tecnologia.
    Tópico: {topico}
    Plano do Post:
    {plano_de_post}
    
    Escreva um rascunho de post para o Instagram com tom leve e amigável,
    utilizando emojis e finalizando com 2 a 4 hashtags relevantes.
    """
    
    return chamar_gemini_com_retry(contents=prompt)

### Agente 4: Revisor de Qualidade

In [6]:
def agente_revisor(topico: str, rascunho_gerado: str) -> str:
    """Faz a revisão final de tom de voz, clareza e gramática."""
    prompt = f"""
    Você é um Editor e Revisor de Conteúdo sênior para redes sociais.
    Público-alvo: Jovens e entusiastas de tecnologia (18 a 30 anos).
    
    Tópico: {topico}
    Rascunho:
    {rascunho_gerado}
    
    Revise a clareza, concisão e correção gramatical do texto.
    Retorne o texto final aprimorado e pronto para publicação.
    """
    
    return chamar_gemini_com_retry(contents=prompt)

## 3. Execução da Pipeline Multi-Agente

In [7]:
# Exemplo de execução da pipeline completa
topico_estudo = "Situação do Corinthians"
data_atual = date.today().strftime("%d/%m/%Y")

print(f"🚀 Iniciando Sistema Multi-Agente para: '{topico_estudo}'\n")

# Step 1: Pesquisa de Notícias
print("[1/4] Buscando novidades...")
pesquisa = agente_buscador(topico_estudo, data_atual)

# Step 2: Planejamento
print("[2/4] Planejando post...")
plano = agente_planejador(topico_estudo, pesquisa)

# Step 3: Redação
print("[3/4] Redigindo rascunho...")
rascunho = agente_redator(topico_estudo, plano)

# Step 4: Revisão Final
print("[4/4] Revisando post...")
post_final = agente_revisor(topico_estudo, rascunho)

print("\n================ RESULTADO FINAL ================\n")
print(post_final)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


🚀 Iniciando Sistema Multi-Agente para: 'Situação do Corinthians'

[1/4] Buscando novidades...
[2/4] Planejando post...
[3/4] Redigindo rascunho...
[4/4] Revisando post...

================ RESULTADO FINAL ================

Excelente rascunho! Ele já capta bem o tom e as informações que o público-alvo busca. Como seu Editor Sênior, vamos lapidar a clareza, concisão e corrigir alguns detalhes para deixá-lo impecável para o Instagram.

Alguns pontos que observei e ajustarei:
*   **Consistência de Voz:** Em alguns momentos, o texto usa "nós" ("já estamos", "levamos", "ocupamos"), o que pode soar como se o editor fosse parte do time. Manteremos a voz de um observador/analista sênior.
*   **Concisão:** Pequenos ajustes para deixar as frases mais diretas, sem perder o charme e a informalidade.
*   **Revisão Pontual:** Pequenas correções gramaticais e de pontuação.
*   **Fluidez:** Otimizar algumas transições.

---

Aqui está o texto aprimorado, pronto para publicação:

---

**Corinthians em 2